In [ ]:
import sys, os, glob as _g, subprocess
try:
    import onnxruntime; print(f'ort {onnxruntime.__version__} ready')
except ImportError:
    wdirs = {os.path.dirname(w) for w in _g.glob('/kaggle/input/**/*.whl', recursive=True)
             if 'onnxruntime' in os.path.basename(w)}
    if not wdirs: raise RuntimeError('no onnxruntime wheel in /kaggle/input')
    fl = [x for d in wdirs for x in ['--find-links', d]]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', *fl, 'onnxruntime'], check=True)
    import onnxruntime; print(f'ort {onnxruntime.__version__} installed')


# BirdCLEF 2026 Inference v26 — v23 GRU + v25 Mel Ensemble

**LB scores** (used to pick weights):
| Branch | LB |
|--------|----|
| v23 GRU-only | **0.858** |
| v25 mel-only (OOF 0.945) | 0.741 |

Mel OOF does NOT translate to LB. Default `gru_weight=0.8, mel_weight=0.2`.  
Weight sweep: `1.0/0.0` → `0.8/0.2` → `0.7/0.3`  

**Required datasets**: birdclef-2026, birdclef-2026-perch-onnx,
birdclef-2026-perch-weights-v23, birdclef-2026-weights-v25


In [ ]:
import os, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa, onnxruntime as ort
from scipy.ndimage import gaussian_filter1d
import torch, torch.nn as nn, torch.nn.functional as F
from torch.cuda.amp import autocast
import timm
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    folds=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    perch_sr=32000, perch_seconds=5, perch_emb_dim=1536, perch_batch=16,
    gru_hidden=512,   # v23 arch -- do NOT change to 768
    gru_layers=2,     # v23 arch
    mel_sr=16000, mel_seconds=5,
    n_mels=64, n_fft=1024, hop_length=320, fmin=60, fmax=8000,
    mel_tta=3,         # 1=no TTA, 3=start/center/end
    gauss_sigma=1.0,
    # LB: GRU-only=0.858, mel-only=0.741
    # mel OOF 0.945 does NOT transfer to LB -> GRU must dominate
    # Recommended sweep: 1.0/0.0 -> 0.8/0.2 -> 0.7/0.3
    gru_weight=0.8,
    mel_weight=0.2,
)
CFG['perch_target'] = CFG['perch_sr'] * CFG['perch_seconds']  # 160000
CFG['mel_target']   = CFG['mel_sr']  * CFG['mel_seconds']     # 80000
device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device: {device}  ort: {ort.__version__}')
print(f'GRU v23  hidden={CFG["gru_hidden"]}  layers={CFG["gru_layers"]}')
print(f'Weights  gru={CFG["gru_weight"]}  mel={CFG["mel_weight"]}  tta={CFG["mel_tta"]}')


In [ ]:
def _fe(*c):
    return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                   '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
TEST_AUDIO   = _fe('/kaggle/input/birdclef-2026/test_soundscapes',
                   '/kaggle/input/competitions/birdclef-2026/test_soundscapes')
SAMPLE_SUB   = _fe('/kaggle/input/birdclef-2026/sample_submission.csv',
                   '/kaggle/input/competitions/birdclef-2026/sample_submission.csv')
# v23 GRU weights
GRU_CKPT_DIR = _fe('/kaggle/input/birdclef-2026-perch-weights-v23',
                   '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v23',
                   '/kaggle/working')
# v25 mel weights
MEL_CKPT_DIR = _fe('/kaggle/input/birdclef-2026-weights-v25',
                   '/kaggle/input/datasets/chiragggg/birdclef-2026-weights-v25',
                   '/kaggle/working')

ONNX_PATH = None
for _c in [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/model.onnx',
]:
    if os.path.exists(_c):
        ONNX_PATH = _c
        break

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
n_classes   = len(species)
sp_idx      = {l: i for i, l in enumerate(species)}
print(f'Species: {n_classes}')
print(f'GRU_CKPT_DIR: {GRU_CKPT_DIR}')
print(f'MEL_CKPT_DIR: {MEL_CKPT_DIR}')
print(f'ONNX_PATH   : {ONNX_PATH}')


In [ ]:
class PerchGRU(nn.Module):
    """v23 architecture: hidden=512, layers=2, no AttentionPool."""
    def __init__(self, n_classes, emb_dim=1536, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(512, hidden, n_layers, batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x):
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out


class BirdCLEFModel(nn.Module):
    def __init__(self, arch, n_classes, pretrained=False):
        super().__init__()
        if arch == 'resnet18':
            b = timm.create_model('resnet18', pretrained=pretrained, in_chans=1)
            nf = b.fc.in_features
            b.fc = nn.Identity()
        elif arch == 'efficientnet_b0':
            b = timm.create_model('efficientnet_b0', pretrained=pretrained, in_chans=1)
            nf = b.classifier.in_features
            b.classifier = nn.Identity()
        else:
            raise ValueError(arch)
        self.backbone = b
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(nf, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, n_classes)
        )

    def forward(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = self.pool(f).flatten(1)
        return self.head(f)


print('Model classes defined')


In [ ]:
def _load(Cls, names, ckpt_dir, **kw):
    ms = []
    for n in names:
        p = Path(ckpt_dir) / n
        if not p.exists():
            print(f'  MISSING: {p}')
            continue
        m = Cls(**kw).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True))
        m.eval()
        ms.append(m)
        print(f'  OK {n}')
    return ms

print('v23 GRU (hidden=512, layers=2)...')
gru_models = _load(PerchGRU,
    [f'perch_gru_v23_fold{i}.pt' for i in range(CFG['folds'])], GRU_CKPT_DIR,
    n_classes=n_classes, emb_dim=CFG['perch_emb_dim'],
    hidden=CFG['gru_hidden'], n_layers=CFG['gru_layers'])

print('ResNet18 v25...')
resnet_models = _load(BirdCLEFModel,
    [f'resnet18_v25_fold{i}.pt' for i in range(CFG['folds'])], MEL_CKPT_DIR,
    arch='resnet18', n_classes=n_classes)

print('EfficientNet-B0 v25...')
effnet_models = _load(BirdCLEFModel,
    [f'efficientnet_b0_v25_fold{i}.pt' for i in range(CFG['folds'])], MEL_CKPT_DIR,
    arch='efficientnet_b0', n_classes=n_classes)

print(f'Loaded  GRU:{len(gru_models)}/5  ResNet18:{len(resnet_models)}/5  EffNetB0:{len(effnet_models)}/5')


In [ ]:
_sess = None; _inp = None; _eidx = 0; _onnx_ok = False
if ONNX_PATH is None:
    print('ONNX not found -- GRU branch will be disabled')
else:
    try:
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = os.cpu_count() or 4
        _sess = ort.InferenceSession(ONNX_PATH, sess_options=opts,
                                     providers=['CPUExecutionProvider'])
        _inp  = _sess.get_inputs()[0].name
        _out_names = [o.name for o in _sess.get_outputs()]
        ekey  = next((o.name for o in _sess.get_outputs()
                      if o.shape and o.shape[-1] == 1536), _out_names[0])
        _eidx = _out_names.index(ekey)
        _t    = _sess.run(None, {_inp: np.zeros((1, CFG['perch_target']), np.float32)})
        _e    = _t[_eidx]
        if _e.ndim == 3:
            _e = _e.mean(1)
        assert _e.shape[-1] == 1536, f'Expected 1536-d emb, got {_e.shape}'
        _onnx_ok = True
        print(f'ONNX OK: {Path(ONNX_PATH).name}  ekey={ekey}  emb={_e.shape}')
    except Exception as ex:
        print(f'ONNX ERROR: {ex}')


In [ ]:
_mel_fb = librosa.filters.mel(
    sr=CFG['mel_sr'], n_fft=CFG['n_fft'],
    n_mels=CFG['n_mels'], fmin=CFG['fmin'], fmax=CFG['fmax'],
)

def logmel(wav, start=None):
    tgt = CFG['mel_target']
    if start is not None:
        c = wav[start:start + tgt]
        if len(c) < tgt:
            c = np.pad(c, (0, tgt - len(c)))
    else:
        c = np.pad(wav, (0, max(0, tgt - len(wav))))[:tgt]
    S  = np.abs(librosa.stft(c, n_fft=CFG['n_fft'], hop_length=CFG['hop_length'],
                             window='hann', center=True)) ** 2
    lm = np.log1p(_mel_fb @ S).astype(np.float32)
    return (lm - lm.mean()) / (lm.std() + 1e-6)


def tta_batch(wav, n):
    """Return (n, 1, n_mels, T) normalised log-mel batch."""
    tgt, L = CFG['mel_target'], len(wav)
    if L <= tgt or n == 1:
        t = torch.from_numpy(logmel(wav)).float().unsqueeze(0).unsqueeze(0)
        return t.expand(n, -1, -1, -1).contiguous()
    starts = [int(i * (L - tgt) / (n - 1)) for i in range(n)]
    return torch.stack([torch.from_numpy(logmel(wav, s)).float()
                        for s in starts]).unsqueeze(1)


_s = np.sin(2 * np.pi * 440 * np.linspace(0, 5, 80000)).astype(np.float32)
print(f'logmel shape: {logmel(_s).shape}')
print(f'tta_batch   : {tta_batch(_s, CFG["mel_tta"]).shape}')
del _s


In [ ]:
_amp = (device.type == 'cuda')

def _embs(path, ends):
    if not _onnx_ok or not ends:
        return {}
    try:
        y, sr = sf.read(path, always_2d=False)
        if y.ndim == 2:
            y = y.mean(1)
        if sr != CFG['perch_sr']:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=CFG['perch_sr'])
        y = y.astype(np.float32)
    except Exception as e:
        print(f'[W] perch read: {e}')
        return {}
    clips = []
    for es in ends:
        e0 = int(es * CFG['perch_sr'])
        s0 = max(0, e0 - CFG['perch_target'])
        c  = y[s0:e0]
        if len(c) < CFG['perch_target']:
            c = np.pad(c, (0, CFG['perch_target'] - len(c)))
        clips.append(c)
    all_embs = []
    for bi in range(0, len(clips), CFG['perch_batch']):
        B   = np.stack(clips[bi:bi + CFG['perch_batch']])
        out = _sess.run(None, {_inp: B})[_eidx]
        if out.ndim == 3:
            out = out.mean(1)
        all_embs.append(out.astype(np.float32))
    return dict(zip(ends, np.vstack(all_embs)))


def predict_v26(path, ends):
    T         = len(ends)
    gru_preds = []
    mel_mat   = None

    # ---- GRU branch ----
    if gru_models and _onnx_ok and CFG['gru_weight'] > 0:
        em  = _embs(path, ends)
        seq = torch.from_numpy(
            np.stack([em.get(e, np.zeros(CFG['perch_emb_dim'], np.float32)) for e in ends])
        ).float().unsqueeze(0).to(device)  # (1, T, 1536)
        for m in gru_models:
            with torch.inference_mode(), autocast(enabled=_amp):
                gru_preds.append(torch.sigmoid(m(seq).float())[0].cpu().numpy())

    # ---- Mel branch with TTA ----
    all_mel = resnet_models + effnet_models
    if all_mel and CFG['mel_weight'] > 0:
        try:
            y16, sr = sf.read(path, always_2d=False)
            if y16.ndim == 2:
                y16 = y16.mean(1)
            if sr != CFG['mel_sr']:
                y16 = librosa.resample(y16.astype(np.float32), orig_sr=sr, target_sr=CFG['mel_sr'])
            y16 = y16.astype(np.float32)
        except Exception as e:
            print(f'[W] mel read: {e}')
            y16 = None
        if y16 is not None:
            rows = []
            for es in ends:
                e0 = int(es * CFG['mel_sr'])
                s0 = max(0, e0 - CFG['mel_target'])
                mb = tta_batch(y16[s0:e0], CFG['mel_tta']).to(device)
                mp = []
                for m in all_mel:
                    with torch.inference_mode(), autocast(enabled=_amp):
                        mp.append(torch.sigmoid(m(mb).float()).mean(0).cpu().numpy())
                rows.append(np.mean(mp, axis=0))
            mel_mat = np.stack(rows)  # (T, n_classes)

    if not gru_preds and mel_mat is None:
        return np.full((T, n_classes), 0.5, np.float32)

    # ---- Weighted ensemble ----
    wsum = np.zeros((T, n_classes), np.float32)
    wt   = 0.0
    if gru_preds:
        wsum += CFG['gru_weight'] * np.mean(gru_preds, axis=0)
        wt   += CFG['gru_weight']
    if mel_mat is not None:
        wsum += CFG['mel_weight'] * mel_mat
        wt   += CFG['mel_weight']
    p = (wsum / wt).astype(np.float32)

    # ---- Gaussian smoothing ----
    if T > 1 and CFG['gauss_sigma'] > 0:
        p = gaussian_filter1d(p.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0).astype(np.float32)
    return p


print(f'predict_v26 defined  GRU={len(gru_models)}  ResNet18={len(resnet_models)}  EffB0={len(effnet_models)}')


In [ ]:
sub = pd.read_csv(SAMPLE_SUB).copy()
sub['_sc'] = sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Rows: {len(sub)}')

row_ids = []; probs_list = []; n_miss = 0; n_err = 0

for sc, grp in tqdm(sub.groupby('_sc'), desc='soundscapes', unit='f'):
    rids = [str(r) for r in grp['row_id']]
    ap = None
    for ext in ['.ogg', '.wav', '.flac']:
        c = Path(TEST_AUDIO) / f'{sc}{ext}'
        if c.exists():
            ap = str(c)
            break
    if ap is None:
        n_miss += 1
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))
        continue
    try:
        ends = [int(r.rsplit('_', 1)[-1]) for r in rids]
    except Exception:
        n_err += 1
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))
        continue
    try:
        p = predict_v26(ap, ends)
        row_ids.extend(rids)
        probs_list.append(p)
    except Exception as e:
        n_err += 1
        print(f'ERR {sc}: {e}')
        row_ids.extend(rids)
        probs_list.append(np.full((len(rids), n_classes), 0.5, np.float32))

print(f'Done  missing={n_miss}  errors={n_err}')


In [ ]:
mat = np.concatenate(probs_list, axis=0)
mu, sd = mat.mean(), mat.std()
if abs(mu - 0.5) < 0.001 and sd < 0.01:
    print(f'WARNING: all-neutral predictions (mean={mu:.4f}, std={sd:.4f})')
    print('  Check Cell 6 (ONNX) and Cell 5 (checkpoints).')
else:
    print(f'OK  mean={mu:.4f}  std={sd:.4f}')

sub_df = pd.DataFrame(mat, columns=species)
sub_df.insert(0, 'row_id', row_ids)
cols   = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[cols]
sub_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved  shape={sub_df.shape}')
print(f'gru={CFG["gru_weight"]}/mel={CFG["mel_weight"]}  '
      f'tta={CFG["mel_tta"]}  '
      f'folds gru={len(gru_models)} resnet={len(resnet_models)} effnet={len(effnet_models)}')
sub_df.head(3)
